<a href="https://colab.research.google.com/github/24071a6236-jpg/polymathai/blob/main/Testing(M5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Recreate the model storage dictionary
trained_models = {}
print("Model storage recreated.")

Model storage recreated.


In [ ]:
import os

print("Current files:")
for f in os.listdir():
    print(f)

Current files:
.config
sample_data


In [ ]:
# ============================================================
# MILESTONE 3 RECOVERY
# Recreate all 6 processed dataset files
# ============================================================

import os
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA

SEEDS = [42, 1337, 2024]

os.makedirs("milestone3_processed", exist_ok=True)

# ============================================================
# LOAD DATASETS
# ============================================================

print("Loading ToN-IoT...")
toniot = load_dataset("codymlewis/TON_IoT_network")
ton_train = toniot["train"].to_pandas()

print("ToN-IoT:", ton_train.shape)

print("\nLoading NSL-KDD...")
nslkdd = load_dataset("Mireu-Lab/NSL-KDD")
nsl_train = nslkdd["train"].to_pandas()

print("NSL-KDD:", nsl_train.shape)


# ============================================================
# PREPROCESSING FUNCTION
# ============================================================

def preprocess_and_save(df, dataset_name, target_column):

    print("\n" + "=" * 60)
    print(dataset_name)
    print("=" * 60)

    X = df.drop(columns=[target_column]).copy()
    y = df[target_column].copy()

    # --------------------------------------------------------
    # Binary labels
    # 0 = Normal
    # 1 = Attack
    # --------------------------------------------------------

    if dataset_name == "NSL-KDD":

        y = (
            y.astype(str)
             .str.lower()
             .eq("normal")
             .astype(int)
        )

        # normal=True becomes 1, so invert:
        y = 1 - y

    else:
        # ToN-IoT already uses:
        # 0 = normal
        # 1 = attack
        y = pd.to_numeric(y).astype(int)

    # --------------------------------------------------------
    # Encode categorical columns
    # --------------------------------------------------------

    categorical_cols = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    print("Categorical columns:", len(categorical_cols))

    if categorical_cols:

        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

        X[categorical_cols] = encoder.fit_transform(
            X[categorical_cols].astype(str)
        )

    # Convert everything to numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    # Replace invalid values
    X = X.replace([np.inf, -np.inf], np.nan)

    X = X.fillna(0)

    X = X.astype(float)

    # --------------------------------------------------------
    # PCA + train/validation split for each seed
    # --------------------------------------------------------

    for seed in SEEDS:

        print(f"\n---------- Seed {seed} ----------")

        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=0.30,
            stratify=y,
            random_state=seed
        )

        # ----------------------------------------------------
        # Standard scaling — fit ONLY on training data
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # ----------------------------------------------------
        # PCA — 8 components
        # ----------------------------------------------------

        pca = PCA(
            n_components=8,
            random_state=seed
        )

        X_train_pca = pca.fit_transform(X_train_scaled)
        X_val_pca = pca.transform(X_val_scaled)

        print("X_train:", X_train_pca.shape)
        print("X_val  :", X_val_pca.shape)
        print("y_train:", y_train.shape)
        print("y_val  :", y_val.shape)

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        output_path = (
            f"milestone3_processed/"
            f"{dataset_name}_seed_{seed}.npz"
        )

        np.savez(
            output_path,
            X_train=X_train_pca,
            X_val=X_val_pca,
            y_train=np.asarray(y_train),
            y_val=np.asarray(y_val)
        )

        print("✅ Saved:", output_path)


# ============================================================
# PROCESS ToN-IoT
# ============================================================

preprocess_and_save(
    ton_train,
    "ToN-IoT",
    "label"
)


# ============================================================
# PROCESS NSL-KDD
# ============================================================

preprocess_and_save(
    nsl_train,
    "NSL-KDD",
    "class"
)


print("\n" + "=" * 60)
print("🎯 MILESTONE 3 FILE CREATION COMPLETE")
print("=" * 60)

Loading ToN-IoT...


README.md:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

train_test_network.csv: reconstructing file:   0%|          |  0.00B / 29.9MB            

train_test_network.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/211043 [00:00<?, ? examples/s]

...

train.csv: downloading bytes:           |  0.00B            

test.csv:   0%|          | 0.00/5.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/151165 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/34394 [00:00<?, ? examples/s]

NSL-KDD: (151165, 42)

ToN-IoT
Categorical columns: 27

---------- Seed 42 ----------
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)
✅ Saved: milestone3_processed/ToN-IoT_seed_42.npz

---------- Seed 1337 ----------
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)
✅ Saved: milestone3_processed/ToN-IoT_seed_1337.npz

---------- Seed 2024 ----------
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)
✅ Saved: milestone3_processed/ToN-IoT_seed_2024.npz

NSL-KDD
Categorical columns: 3

---------- Seed 42 ----------
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)
✅ Saved: milestone3_processed/NSL-KDD_seed_42.npz

---------- Seed 1337 ----------
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)
✅ Saved: milestone3_processed/NSL-KDD_seed_1337.npz

---------- Seed 2024 ----------
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (453

In [ ]:
# ============================================================
# MILESTONE 3 — FINAL VERIFICATION
# ============================================================

import os
import numpy as np

SEEDS = [42, 1337, 2024]
DATASETS = ["NSL-KDD", "ToN-IoT"]

print("Checking all processed files...\n")

all_ok = True

for dataset in DATASETS:
    for seed in SEEDS:

        path = f"milestone3_processed/{dataset}_seed_{seed}.npz"

        if not os.path.exists(path):
            print(f"❌ MISSING: {path}")
            all_ok = False
            continue

        data = np.load(path)

        required = ["X_train", "X_val", "y_train", "y_val"]

        if not all(key in data.files for key in required):
            print(f"❌ Missing arrays: {path}")
            all_ok = False
            continue

        print(
            f"✅ {dataset} | Seed {seed} | "
            f"Train {data['X_train'].shape} | "
            f"Val {data['X_val'].shape}"
        )

if all_ok:
    print("\n🎯 ALL 6 PROCESSED FILES VERIFIED.")
else:
    print("\n❌ Verification failed.")

Checking all processed files...

✅ NSL-KDD | Seed 42 | Train (105815, 8) | Val (45350, 8)
✅ NSL-KDD | Seed 1337 | Train (105815, 8) | Val (45350, 8)
✅ NSL-KDD | Seed 2024 | Train (105815, 8) | Val (45350, 8)
✅ ToN-IoT | Seed 42 | Train (147730, 8) | Val (63313, 8)
✅ ToN-IoT | Seed 1337 | Train (147730, 8) | Val (63313, 8)
✅ ToN-IoT | Seed 2024 | Train (147730, 8) | Val (63313, 8)

🎯 ALL 6 PROCESSED FILES VERIFIED.


In [ ]:
# ============================================================
# RESTORE QISKIT ENVIRONMENT
# ============================================================

!pip install -q qiskit==2.5.2 qiskit-machine-learning==0.9.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.3/282.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.8 MB/s eta 0:00:00


In [ ]:
import qiskit
import qiskit_machine_learning

print("Qiskit version:", qiskit.__version__)
print("Qiskit Machine Learning version:", qiskit_machine_learning.__version__)

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, VQC

print("✅ All Qiskit imports successful.")

Qiskit version: 2.5.2
Qiskit Machine Learning version: 0.9.1
✅ All Qiskit imports successful.


In [ ]:
# ============================================================
# MILESTONE 4 — TRAIN QSVC, VQC, QNN
# ============================================================

import numpy as np
import pandas as pd
import os
import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, VQC
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

SEEDS = [42, 1337, 2024]
DATASETS = ["NSL-KDD", "ToN-IoT"]
METHODS = ["QSVC", "VQC", "QNN"]

TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100

N_QUBITS = 8

trained_models = {}

print("============================================================")
print("MILESTONE 4 — TRAINING QSVC, VQC, QNN")
print("============================================================")
print(f"Training samples : {TRAIN_SAMPLES}")
print(f"Evaluation samples: {EVAL_SAMPLES}")
print(f"Qubits/features  : {N_QUBITS}")
print(f"Seeds            : {SEEDS}")
print()

MILESTONE 4 — TRAINING QSVC, VQC, QNN
Training samples : 100
Evaluation samples: 100
Qubits/features  : 8
Seeds            : [42, 1337, 2024]



In [ ]:
# ============================================================
# MILESTONE 4 — QSVC TRAINING FUNCTION
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

SEEDS = [42, 1337, 2024]
TRAIN_SAMPLES = 100


def train_qsvc(X_train, y_train, seed):

    # 1. Stratified training subset
    X_subset, _, y_subset, _ = train_test_split(
        X_train,
        y_train,
        train_size=TRAIN_SAMPLES,
        stratify=y_train,
        random_state=seed
    )

    # 2. Map PCA features to [0, pi]
    angle_scaler = MinMaxScaler(
        feature_range=(0, np.pi)
    )

    X_subset_scaled = angle_scaler.fit_transform(
        X_subset
    )

    # 3. Create 8-qubit ZZ feature map
    feature_map = zz_feature_map(
        feature_dimension=8,
        reps=1,
        entanglement="linear"
    )

    # 4. Fidelity quantum kernel
    quantum_kernel = FidelityQuantumKernel(
        feature_map=feature_map
    )

    # 5. QSVC
    qsvc = QSVC(
        quantum_kernel=quantum_kernel
    )

    # 6. Train
    qsvc.fit(
        X_subset_scaled,
        y_subset
    )

    return qsvc, angle_scaler, X_subset, y_subset


print("✅ QSVC training function created.")
print("Training samples:", TRAIN_SAMPLES)
print("Features / qubits:", 8)
print("Seeds:", SEEDS)

✅ QSVC training function created.
Training samples: 100
Features / qubits: 8
Seeds: [42, 1337, 2024]


In [ ]:
# ============================================================
# MILESTONE 4 — RUN 6 QSVC MODELS
# ============================================================

import numpy as np
import time
from sklearn.metrics import accuracy_score, f1_score

qsvc_models = {}
qsvc_results = []

for dataset in ["NSL-KDD", "ToN-IoT"]:

    for seed in [42, 1337, 2024]:

        print("\n" + "=" * 60)
        print(f"QSVC | {dataset} | Seed {seed}")
        print("=" * 60)

        # Load Milestone 3 data
        path = f"milestone3_processed/{dataset}_seed_{seed}.npz"
        data = np.load(path)

        X_train = data["X_train"]
        y_train = data["y_train"]

        X_val = data["X_val"]
        y_val = data["y_val"]

        print("Full training :", X_train.shape)
        print("Validation    :", X_val.shape)

        # Evaluation subset: 100 samples, stratified
        X_eval, _, y_eval, _ = train_test_split(
            X_val,
            y_val,
            train_size=100,
            stratify=y_val,
            random_state=seed + 10000
        )

        start = time.time()

        # Train QSVC
        qsvc, scaler, X_subset, y_subset = train_qsvc(
            X_train,
            y_train,
            seed
        )

        training_time = time.time() - start

        # Clean evaluation
        X_eval_scaled = scaler.transform(X_eval)

        pred_start = time.time()

        y_pred = qsvc.predict(X_eval_scaled)

        prediction_time = time.time() - pred_start

        accuracy = accuracy_score(
            y_eval,
            y_pred
        )

        macro_f1 = f1_score(
            y_eval,
            y_pred,
            average="macro"
        )

        # Store model
        qsvc_models[(dataset, seed)] = {
            "model": qsvc,
            "scaler": scaler,
            "X_eval": X_eval.copy(),
            "y_eval": y_eval.copy(),
            "train_std": X_train.std(axis=0)
        }

        qsvc_results.append({
            "Dataset": dataset,
            "Model": "QSVC",
            "Seed": seed,
            "Accuracy": accuracy,
            "MacroF1": macro_f1,
            "TrainingTimeSec": training_time,
            "PredictionTimeSec": prediction_time
        })

        print(f"Training time : {training_time:.2f} sec")
        print(f"Prediction    : {prediction_time:.2f} sec")
        print(f"Accuracy      : {accuracy:.4f}")
        print(f"Macro-F1      : {macro_f1:.4f}")

print("\n" + "=" * 60)
print("QSVC TRAINING COMPLETE")
print("=" * 60)

print("QSVC models stored:", len(qsvc_models), "/ 6")

qsvc_results_df = pd.DataFrame(qsvc_results)
display(qsvc_results_df)


QSVC | NSL-KDD | Seed 42
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 226.42 sec
Prediction    : 342.54 sec
Accuracy      : 0.8800
Macro-F1      : 0.8760

QSVC | NSL-KDD | Seed 1337
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 169.54 sec
Prediction    : 334.92 sec
Accuracy      : 0.9000
Macro-F1      : 0.8974

QSVC | NSL-KDD | Seed 2024
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 166.07 sec
Prediction    : 331.67 sec
Accuracy      : 0.9100
Macro-F1      : 0.9085

QSVC | ToN-IoT | Seed 42
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 165.32 sec
Prediction    : 333.39 sec
Accuracy      : 0.9700
Macro-F1      : 0.9570

QSVC | ToN-IoT | Seed 1337
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 164.01 sec
Prediction    : 331.36 sec
Accuracy      : 0.9000
Macro-F1      : 0.8376

QSVC | ToN-IoT | Seed 2024
Full training : (147730, 8)
Validation    : (63313, 8)
Traini

,Dataset,Model,Seed,Accuracy,MacroF1,TrainingTimeSec,PredictionTimeSec
0,NSL-KDD,QSVC,42,0.88,0.875982,226.416712,342.538004
1,NSL-KDD,QSVC,1337,0.90,0.897373,169.544445,334.918352
2,NSL-KDD,QSVC,2024,0.91,0.908453,166.066416,331.665251
3,ToN-IoT,QSVC,42,0.97,0.956989,165.324316,333.390434
4,ToN-IoT,QSVC,1337,0.90,0.837557,164.013513,331.363710
5,ToN-IoT,QSVC,2024,0.96,0.941725,164.049196,357.233428


In [ ]:
# ============================================================
# MILESTONE 4 — VQC TRAINING FUNCTION
# ============================================================

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA

SEEDS = [42, 1337, 2024]

TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100
N_QUBITS = 8


def train_vqc(X_train, y_train, seed):

    # --------------------------------------------------------
    # 1. Select 100 stratified training samples
    # --------------------------------------------------------
    X_subset, _, y_subset, _ = train_test_split(
        X_train,
        y_train,
        train_size=TRAIN_SAMPLES,
        stratify=y_train,
        random_state=seed
    )

    # --------------------------------------------------------
    # 2. Scale PCA features to [0, pi]
    # --------------------------------------------------------
    scaler = MinMaxScaler(
        feature_range=(0, np.pi)
    )

    X_subset_scaled = scaler.fit_transform(
        X_subset
    )

    # --------------------------------------------------------
    # 3. Create 8-qubit ZZ feature map
    # --------------------------------------------------------
    feature_map = zz_feature_map(
        feature_dimension=N_QUBITS,
        reps=1,
        entanglement="linear"
    )

    # --------------------------------------------------------
    # 4. Create RealAmplitudes ansatz
    # --------------------------------------------------------
    ansatz = real_amplitudes(
        num_qubits=N_QUBITS,
        reps=1,
        entanglement="linear"
    )

    # --------------------------------------------------------
    # 5. COBYLA optimizer
    # --------------------------------------------------------
    optimizer = COBYLA(
        maxiter=30
    )

    # --------------------------------------------------------
    # 6. Create VQC
    # --------------------------------------------------------
    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=optimizer
    )

    # --------------------------------------------------------
    # 7. Train
    # --------------------------------------------------------
    start = time.time()

    vqc.fit(
        X_subset_scaled,
        y_subset
    )

    training_time = time.time() - start

    return (
        vqc,
        scaler,
        X_subset,
        y_subset,
        training_time
    )


print("✅ VQC training function created.")
print("Training samples :", TRAIN_SAMPLES)
print("Evaluation samples:", EVAL_SAMPLES)
print("Features / qubits:", N_QUBITS)
print("Seeds:", SEEDS)

✅ VQC training function created.
Training samples : 100
Evaluation samples: 100
Features / qubits: 8
Seeds: [42, 1337, 2024]


In [ ]:
# ============================================================
# MILESTONE 4 — RUN 6 VQC MODELS
# ============================================================

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

vqc_models = {}
vqc_results = []

for dataset in ["NSL-KDD", "ToN-IoT"]:

    for seed in [42, 1337, 2024]:

        print("\n" + "=" * 60)
        print(f"VQC | {dataset} | Seed {seed}")
        print("=" * 60)

        # ----------------------------------------------------
        # 1. Load processed dataset
        # ----------------------------------------------------
        path = f"milestone3_processed/{dataset}_seed_{seed}.npz"

        data = np.load(path)

        X_train = data["X_train"]
        y_train = data["y_train"]

        X_val = data["X_val"]
        y_val = data["y_val"]

        print("Full training :", X_train.shape)
        print("Validation    :", X_val.shape)

        # ----------------------------------------------------
        # 2. Select 100 evaluation samples
        # ----------------------------------------------------
        X_eval, _, y_eval, _ = train_test_split(
            X_val,
            y_val,
            train_size=100,
            stratify=y_val,
            random_state=seed
        )

        # ----------------------------------------------------
        # 3. Train VQC
        # ----------------------------------------------------
        (
            vqc,
            scaler,
            X_subset,
            y_subset,
            training_time
        ) = train_vqc(
            X_train,
            y_train,
            seed
        )

        # ----------------------------------------------------
        # 4. Clean evaluation
        # ----------------------------------------------------
        X_eval_scaled = scaler.transform(
            X_eval
        )

        pred_start = time.time()

        y_pred = vqc.predict(
            X_eval_scaled
        )

        prediction_time = time.time() - pred_start

        # ----------------------------------------------------
        # 5. Metrics
        # ----------------------------------------------------
        accuracy = accuracy_score(
            y_eval,
            y_pred
        )

        macro_f1 = f1_score(
            y_eval,
            y_pred,
            average="macro"
        )

        # ----------------------------------------------------
        # 6. Store model
        # ----------------------------------------------------
        vqc_models[(dataset, seed)] = {
            "model": vqc,
            "scaler": scaler,
            "X_eval": X_eval.copy(),
            "y_eval": y_eval.copy(),
            "train_std": X_train.std(axis=0)
        }

        # ----------------------------------------------------
        # 7. Store results
        # ----------------------------------------------------
        vqc_results.append({
            "Dataset": dataset,
            "Model": "VQC",
            "Seed": seed,
            "Accuracy": accuracy,
            "MacroF1": macro_f1,
            "TrainingTimeSec": training_time,
            "PredictionTimeSec": prediction_time
        })

        # ----------------------------------------------------
        # 8. Display result
        # ----------------------------------------------------
        print(f"Training time : {training_time / 60:.2f} min")
        print(f"Prediction    : {prediction_time:.2f} sec")
        print(f"Accuracy      : {accuracy:.4f}")
        print(f"Macro-F1      : {macro_f1:.4f}")


print("\n" + "=" * 60)
print("VQC TRAINING COMPLETE")
print("=" * 60)

print(
    "VQC models stored:",
    len(vqc_models),
    "/ 6"
)

vqc_results_df = pd.DataFrame(
    vqc_results
)

display(vqc_results_df)


VQC | NSL-KDD | Seed 42
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 1.55 min
Prediction    : 3.74 sec
Accuracy      : 0.6400
Macro-F1      : 0.6364

VQC | NSL-KDD | Seed 1337
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 1.54 min
Prediction    : 2.72 sec
Accuracy      : 0.5800
Macro-F1      : 0.5625

VQC | NSL-KDD | Seed 2024
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 1.51 min
Prediction    : 2.74 sec
Accuracy      : 0.7000
Macro-F1      : 0.6956

VQC | ToN-IoT | Seed 42
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 1.52 min
Prediction    : 2.77 sec
Accuracy      : 0.5700
Macro-F1      : 0.5305

VQC | ToN-IoT | Seed 1337
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 1.53 min
Prediction    : 3.36 sec
Accuracy      : 0.8300
Macro-F1      : 0.7482

VQC | ToN-IoT | Seed 2024
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 1.57 min
Predict

,Dataset,Model,Seed,Accuracy,MacroF1,TrainingTimeSec,PredictionTimeSec
0,NSL-KDD,VQC,42,0.64,0.636364,93.137104,3.743212
1,NSL-KDD,VQC,1337,0.58,0.562500,92.419589,2.716424
2,NSL-KDD,VQC,2024,0.70,0.695617,90.823400,2.743157
3,ToN-IoT,VQC,42,0.57,0.530516,91.458081,2.767354
4,ToN-IoT,VQC,1337,0.83,0.748185,91.532425,3.358573
5,ToN-IoT,VQC,2024,0.70,0.635746,94.173501,2.709751


In [ ]:
# ============================================================
# MILESTONE 4 — QNN TRAINING FUNCTION
# ============================================================

import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit.circuit.library import zz_feature_map, real_amplitudes

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

SEEDS = [42, 1337, 2024]

TRAIN_SAMPLES = 100
EVAL_SAMPLES = 100
N_QUBITS = 8


def train_qnn(X_train, y_train, seed):

    # --------------------------------------------------------
    # 1. Select 100 stratified training samples
    # --------------------------------------------------------
    X_subset, _, y_subset, _ = train_test_split(
        X_train,
        y_train,
        train_size=TRAIN_SAMPLES,
        stratify=y_train,
        random_state=seed
    )

    # --------------------------------------------------------
    # 2. Scale features to [0, pi]
    # --------------------------------------------------------
    scaler = MinMaxScaler(
        feature_range=(0, np.pi)
    )

    X_subset_scaled = scaler.fit_transform(
        X_subset
    )

    # --------------------------------------------------------
    # 3. Quantum feature map
    # --------------------------------------------------------
    feature_map = zz_feature_map(
        feature_dimension=N_QUBITS,
        reps=1,
        entanglement="linear"
    )

    # --------------------------------------------------------
    # 4. Variational ansatz
    # --------------------------------------------------------
    ansatz = real_amplitudes(
        num_qubits=N_QUBITS,
        reps=1,
        entanglement="linear"
    )

    # --------------------------------------------------------
    # 5. QNN
    # --------------------------------------------------------
    qnn = EstimatorQNN(
        circuit=feature_map.compose(ansatz),
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters
    )

    # --------------------------------------------------------
    # 6. Optimizer
    # --------------------------------------------------------
    optimizer = COBYLA(
        maxiter=30
    )

    # --------------------------------------------------------
    # 7. Neural Network Classifier
    # --------------------------------------------------------
    qnn_classifier = NeuralNetworkClassifier(
        neural_network=qnn,
        optimizer=optimizer
    )

    # --------------------------------------------------------
    # 8. ACTUAL QNN TRAINING
    # --------------------------------------------------------
    start = time.time()

    qnn_classifier.fit(
        X_subset_scaled,
        y_subset
    )

    training_time = time.time() - start

    return (
        qnn_classifier,
        scaler,
        X_subset,
        y_subset,
        training_time
    )


print("✅ QNN training function created.")
print("Training samples :", TRAIN_SAMPLES)
print("Evaluation samples:", EVAL_SAMPLES)
print("Features / qubits:", N_QUBITS)
print("Seeds:", SEEDS)

✅ QNN training function created.
Training samples : 100
Evaluation samples: 100
Features / qubits: 8
Seeds: [42, 1337, 2024]


In [ ]:
# ============================================================
# MILESTONE 4 — RUN 6 QNN MODELS
# ============================================================

import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

qnn_models = {}
qnn_results = []

for dataset in ["NSL-KDD", "ToN-IoT"]:

    for seed in [42, 1337, 2024]:

        print("\n" + "=" * 60)
        print(f"QNN | {dataset} | Seed {seed}")
        print("=" * 60)

        # ----------------------------------------------------
        # 1. Load processed dataset
        # ----------------------------------------------------
        path = f"milestone3_processed/{dataset}_seed_{seed}.npz"

        data = np.load(path)

        X_train = data["X_train"]
        y_train = data["y_train"]

        X_val = data["X_val"]
        y_val = data["y_val"]

        print("Full training :", X_train.shape)
        print("Validation    :", X_val.shape)

        # ----------------------------------------------------
        # 2. Select evaluation samples
        # ----------------------------------------------------
        X_eval, _, y_eval, _ = train_test_split(
            X_val,
            y_val,
            train_size=100,
            stratify=y_val,
            random_state=seed
        )

        # ----------------------------------------------------
        # 3. Train QNN
        # ----------------------------------------------------
        (
            qnn_classifier,
            scaler,
            X_subset,
            y_subset,
            training_time
        ) = train_qnn(
            X_train,
            y_train,
            seed
        )

        # ----------------------------------------------------
        # 4. Clean evaluation
        # ----------------------------------------------------
        X_eval_scaled = scaler.transform(
            X_eval
        )

        pred_start = time.time()

        y_pred = qnn_classifier.predict(
            X_eval_scaled
        )

        prediction_time = time.time() - pred_start

        # ----------------------------------------------------
        # 5. Metrics
        # ----------------------------------------------------
        accuracy = accuracy_score(
            y_eval,
            y_pred
        )

        macro_f1 = f1_score(
            y_eval,
            y_pred,
            average="macro"
        )

        # ----------------------------------------------------
        # 6. Store trained model
        # ----------------------------------------------------
        qnn_models[(dataset, seed)] = {
            "model": qnn_classifier,
            "scaler": scaler,
            "X_eval": X_eval.copy(),
            "y_eval": y_eval.copy(),
            "train_std": X_train.std(axis=0)
        }

        # ----------------------------------------------------
        # 7. Store results
        # ----------------------------------------------------
        qnn_results.append({
            "Dataset": dataset,
            "Model": "QNN",
            "Seed": seed,
            "Accuracy": accuracy,
            "MacroF1": macro_f1,
            "TrainingTimeSec": training_time,
            "PredictionTimeSec": prediction_time
        })

        # ----------------------------------------------------
        # 8. Display result
        # ----------------------------------------------------
        print(f"Training time : {training_time / 60:.2f} min")
        print(f"Prediction    : {prediction_time:.2f} sec")
        print(f"Accuracy      : {accuracy:.4f}")
        print(f"Macro-F1      : {macro_f1:.4f}")


print("\n" + "=" * 60)
print("QNN TRAINING COMPLETE")
print("=" * 60)

print(
    "QNN models stored:",
    len(qnn_models),
    "/ 6"
)

qnn_results_df = pd.DataFrame(
    qnn_results
)

display(qnn_results_df)


QNN | NSL-KDD | Seed 42
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 0.30 min
Prediction    : 0.60 sec
Accuracy      : 0.2200
Macro-F1      : 0.1438

QNN | NSL-KDD | Seed 1337
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 0.23 min
Prediction    : 0.50 sec
Accuracy      : 0.4000
Macro-F1      : 0.2360

QNN | NSL-KDD | Seed 2024
Full training : (105815, 8)
Validation    : (45350, 8)
Training time : 0.23 min
Prediction    : 0.40 sec
Accuracy      : 0.3900
Macro-F1      : 0.2549

QNN | ToN-IoT | Seed 42
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 0.23 min
Prediction    : 0.43 sec
Accuracy      : 0.6400
Macro-F1      : 0.2807

QNN | ToN-IoT | Seed 1337
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 0.23 min
Prediction    : 0.40 sec
Accuracy      : 0.4500
Macro-F1      : 0.2128

QNN | ToN-IoT | Seed 2024
Full training : (147730, 8)
Validation    : (63313, 8)
Training time : 0.24 min
Predict

,Dataset,Model,Seed,Accuracy,MacroF1,TrainingTimeSec,PredictionTimeSec
0,NSL-KDD,QNN,42,0.22,0.143791,17.987976,0.601748
1,NSL-KDD,QNN,1337,0.40,0.235988,13.814370,0.497349
2,NSL-KDD,QNN,2024,0.39,0.254902,13.601489,0.397648
3,ToN-IoT,QNN,42,0.64,0.280702,13.639307,0.432797
4,ToN-IoT,QNN,1337,0.45,0.212766,13.618828,0.398046
5,ToN-IoT,QNN,2024,0.47,0.242894,14.403488,0.397644


In [ ]:
import os
import pickle

os.makedirs("milestone4_saved", exist_ok=True)

with open(
    "milestone4_saved/qsvc_models.pkl",
    "wb"
) as f:
    pickle.dump(qsvc_models, f)

print("✅ QSVC models saved:", len(qsvc_models), "/ 6")

✅ QSVC models saved: 6 / 6


In [ ]:
import os

SAVE_DIR = "milestone4_saved"

print("=" * 60)
print("CHECKING SAVED MILESTONE 4 FILES")
print("=" * 60)

for filename in [
    "qsvc_models.pkl",
    "vqc_models.dill",
    "qnn_models.pkl",
    "qsvc_results.csv",
    "vqc_results.csv",
    "qnn_results.csv"
]:

    path = os.path.join(SAVE_DIR, filename)

    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"✅ {filename:25s} {size:,} bytes")
    else:
        print(f"❌ {filename:25s} NOT FOUND")

CHECKING SAVED MILESTONE 4 FILES
✅ qsvc_models.pkl           170,853 bytes
❌ vqc_models.dill           NOT FOUND
❌ qnn_models.pkl            NOT FOUND
❌ qsvc_results.csv          NOT FOUND
❌ vqc_results.csv           NOT FOUND
❌ qnn_results.csv           NOT FOUND


In [ ]:
with open(
    "milestone4_saved/qnn_models.pkl",
    "wb"
) as f:
    pickle.dump(qnn_models, f)

print("✅ QNN models saved:", len(qnn_models), "/ 6")

✅ QNN models saved: 6 / 6


In [ ]:
qsvc_results_df.to_csv(
    "milestone4_saved/qsvc_results.csv",
    index=False
)

vqc_results_df.to_csv(
    "milestone4_saved/vqc_results.csv",
    index=False
)

qnn_results_df.to_csv(
    "milestone4_saved/qnn_results.csv",
    index=False
)

print("✅ All three result CSVs saved.")

✅ All three result CSVs saved.


In [ ]:
!pip install -q dill

In [ ]:
import dill

with open(
    "milestone4_saved/vqc_models.dill",
    "wb"
) as f:
    dill.dump(vqc_models, f)

print("✅ VQC models saved:", len(vqc_models), "/ 6")

✅ VQC models saved: 6 / 6


In [ ]:
import os

print("=" * 60)
print("MILESTONE 4 SAVE VERIFICATION")
print("=" * 60)

print("\nModels in memory:")
print("QSVC:", len(qsvc_models), "/ 6")
print("VQC :", len(vqc_models), "/ 6")
print("QNN :", len(qnn_models), "/ 6")

print("\nSaved files:")

for filename in sorted(os.listdir("milestone4_saved")):
    print("✅", filename)

MILESTONE 4 SAVE VERIFICATION

Models in memory:
QSVC: 6 / 6
VQC : 6 / 6
QNN : 6 / 6

Saved files:
✅ qnn_models.pkl
✅ qnn_results.csv
✅ qsvc_models.pkl
✅ qsvc_results.csv
✅ vqc_models.dill
✅ vqc_results.csv


In [ ]:
# ============================================================
# MILESTONE 5 — TEST-TIME GAUSSIAN NOISE STRESS TEST
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

NOISE_LEVELS = [0.0, 0.1, 0.3, 0.6]

milestone5_results = []

print("=" * 60)
print("MILESTONE 5 — TEST-TIME GAUSSIAN NOISE STRESS TEST")
print("=" * 60)

print("Noise levels:", NOISE_LEVELS)
print("Methods     :", ["QSVC", "VQC", "QNN"])
print("Datasets    :", ["NSL-KDD", "ToN-IoT"])
print("Seeds       :", [42, 1337, 2024])
print("Expected conditions:", 3 * 2 * 3 * 4)

MILESTONE 5 — TEST-TIME GAUSSIAN NOISE STRESS TEST
Noise levels: [0.0, 0.1, 0.3, 0.6]
Methods     : ['QSVC', 'VQC', 'QNN']
Datasets    : ['NSL-KDD', 'ToN-IoT']
Seeds       : [42, 1337, 2024]
Expected conditions: 72


In [22]:
# ============================================================
# MILESTONE 5 — RUN GAUSSIAN NOISE STRESS TEST
# ============================================================

import numpy as np
import pandas as pd
import timer

from sklearn.metrics import f1_score

# ------------------------------------------------------------
# Combine the 18 trained models
# ------------------------------------------------------------
all_models = {}

for (dataset, seed), info in qsvc_models.items():
    all_models[(dataset, "QSVC", seed)] = info

for (dataset, seed), info in vqc_models.items():
    all_models[(dataset, "VQC", seed)] = info

for (dataset, seed), info in qnn_models.items():
    all_models[(dataset, "QNN", seed)] = info


print("=" * 60)
print("MILESTONE 5 — GAUSSIAN NOISE STRESS TEST")
print("=" * 60)

print("Total trained models:", len(all_models))
print("Expected:", 18)

# ------------------------------------------------------------
# Check that all 18 models are present
# ------------------------------------------------------------
assert len(all_models) == 18, \
    f"Expected 18 models, found {len(all_models)}"

# ------------------------------------------------------------
# Noise levels
# ------------------------------------------------------------
NOISE_LEVELS = [0.0, 0.1, 0.3, 0.6]

milestone5_results = []

# ------------------------------------------------------------
# Fixed offsets for deterministic noise
# Same dataset + seed + noise level
# gives the same noise realization across methods.
# ------------------------------------------------------------
dataset_offset = {
    "NSL-KDD": 100000,
    "ToN-IoT": 200000
}

noise_offset = {
    0.0: 0,
    0.1: 1000,
    0.3: 3000,
    0.6: 6000
}

# ------------------------------------------------------------
# Run all 18 models × 4 noise levels
# ------------------------------------------------------------
condition_number = 0
total_conditions = 18 * len(NOISE_LEVELS)

for (dataset, model_name, seed), info in all_models.items():

    model = info["model"]
    scaler = info["scaler"]

    X_eval = info["X_eval"]
    y_eval = info["y_eval"]

    train_std = info["train_std"]

    print("\n" + "=" * 60)
    print(f"{model_name} | {dataset} | Seed {seed}")
    print("=" * 60)

    for noise_level in NOISE_LEVELS:

        condition_number += 1

        # ----------------------------------------------------
        # 1. Clean case
        # ----------------------------------------------------
        if noise_level == 0.0:

            X_corrupted = X_eval.copy()

        # ----------------------------------------------------
        # 2. Gaussian corruption
        # ----------------------------------------------------
        else:

            rng_seed = (
                dataset_offset[dataset]
                + seed
                + noise_offset[noise_level]
            )

            rng = np.random.default_rng(
                rng_seed
            )

            # Per-feature noise standard deviation
            sigma = noise_level * train_std

            noise = rng.normal(
                loc=0.0,
                scale=sigma,
                size=X_eval.shape
            )

            X_corrupted = X_eval + noise

        # ----------------------------------------------------
        # 3. Apply the model's trained scaler
        # ----------------------------------------------------
        X_input = scaler.transform(
            X_corrupted
        )

        # ----------------------------------------------------
        # 4. Predict
        # ----------------------------------------------------
        start = time.time()

        y_pred = model.predict(
            X_input
        )

        prediction_time = time.time() - start

        # ----------------------------------------------------
        # 5. Macro-F1
        # ----------------------------------------------------
        macro_f1 = f1_score(
            y_eval,
            y_pred,
            average="macro"
        )

        # ----------------------------------------------------
        # 6. Store result
        # ----------------------------------------------------
        milestone5_results.append({
            "Dataset": dataset,
            "Model": model_name,
            "Seed": seed,
            "NoiseLevel": noise_level,
            "MacroF1": macro_f1,
            "PredictionTimeSec": prediction_time
        })

        print(
            f"[{condition_number:02d}/{total_conditions}] "
            f"Noise={noise_level:.1f} | "
            f"Macro-F1={macro_f1:.4f} | "
            f"Time={prediction_time:.2f}s"
        )


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------
milestone5_df = pd.DataFrame(
    milestone5_results
)

print("\n" + "=" * 60)
print("MILESTONE 5 COMPLETE")
print("=" * 60)

print(
    "Total scoring conditions:",
    len(milestone5_df)
)

print(
    "Expected:",
    72
)

display(
    milestone5_df
)

MILESTONE 5 — GAUSSIAN NOISE STRESS TEST
Total trained models: 18
Expected: 18

QSVC | NSL-KDD | Seed 42
[01/72] Noise=0.0 | Macro-F1=0.8760 | Time=359.78s
[02/72] Noise=0.1 | Macro-F1=0.6997 | Time=338.96s
[03/72] Noise=0.3 | Macro-F1=0.4970 | Time=339.14s
[04/72] Noise=0.6 | Macro-F1=0.4283 | Time=338.14s

QSVC | NSL-KDD | Seed 1337
[05/72] Noise=0.0 | Macro-F1=0.8974 | Time=335.61s
[06/72] Noise=0.1 | Macro-F1=0.6260 | Time=336.35s
[07/72] Noise=0.3 | Macro-F1=0.4224 | Time=345.85s
[08/72] Noise=0.6 | Macro-F1=0.4283 | Time=337.96s

QSVC | NSL-KDD | Seed 2024
[09/72] Noise=0.0 | Macro-F1=0.9085 | Time=337.86s
[10/72] Noise=0.1 | Macro-F1=0.8980 | Time=339.24s
[11/72] Noise=0.3 | Macro-F1=0.4543 | Time=335.30s
[12/72] Noise=0.6 | Macro-F1=0.3695 | Time=335.63s

QSVC | ToN-IoT | Seed 42
[13/72] Noise=0.0 | Macro-F1=0.9570 | Time=330.97s
[14/72] Noise=0.1 | Macro-F1=0.5464 | Time=330.74s
[15/72] Noise=0.3 | Macro-F1=0.4846 | Time=325.60s
[16/72] Noise=0.6 | Macro-F1=0.4186 | Time=331.7

,Dataset,Model,Seed,NoiseLevel,MacroF1,PredictionTimeSec
0,NSL-KDD,QSVC,42,0.0,0.875982,359.781283
1,NSL-KDD,QSVC,42,0.1,0.699700,338.964395
2,NSL-KDD,QSVC,42,0.3,0.496994,339.140649
3,NSL-KDD,QSVC,42,0.6,0.428274,338.137803
4,NSL-KDD,QSVC,1337,0.0,0.897373,335.612199
...,...,...,...,...,...,...
67,ToN-IoT,QNN,1337,0.6,0.186667,0.409775
68,ToN-IoT,QNN,2024,0.0,0.242894,0.429300
69,ToN-IoT,QNN,2024,0.1,0.202667,0.414553
70,ToN-IoT,QNN,2024,0.3,0.197333,0.432981
